# 01 · Images and bounding boxes
**Level:** beginner · **Time:** 35–45 minutes · **Runs offline on CPU**

**You will:** inspect pixels, draw annotations, convert box formats, and keep boxes aligned when resizing or flipping an image.

**Prerequisites:** complete notebook 00; understand variables, lists, and indexing. Run cells from top to bottom with **Shift+Enter**. Restart the kernel and run all cells to repeat the demonstration.

**Teaching question:** a classifier answers “what is in this picture?” A detector also answers “where is each object?” Each object needs its own class label and bounding box.


In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "course_utils.py").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Open Jupyter from the ObjectDetection project folder (see README.md).")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from course_utils import seed_everything, show_boxes
seed_everything(42)
print("Project:", ROOT)


## Step 1 — Create an image you can explain
Images are arrays with height, width, and color channels. Our example uses RGB: red, green, blue. NumPy indexes a pixel as `image[y, x]`, even though boxes use `(x, y)` coordinates.

We use **half-open boxes** `[x1, y1, x2, y2]`: the first corner is included and the last corner is excluded. Width is `x2 - x1`; there is no `+1`. The coordinate origin is at the top left; y grows downward.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

image = Image.new("RGB", (360, 240), (235, 240, 245))  # width, height
draw = ImageDraw.Draw(image)
draw.rectangle((35, 45, 134, 164), fill=(225, 70, 55))
draw.ellipse((205, 65, 304, 164), fill=(40, 110, 225))
boxes = np.array([[35, 45, 135, 165], [205, 65, 305, 165]], dtype=np.float32)
names = ["rectangle", "circle"]
pixels = np.asarray(image)
print("Array shape (height, width, channels):", pixels.shape)
print("Storage:", pixels.dtype, "| red object pixel:", pixels[80, 60])
show_boxes(image, boxes, labels=names, title="Two objects, two annotations")
plt.show()


**Observe:** the circle's bounding box includes background corners. A bounding box describes a rectangle around an object; a segmentation mask would describe the object's individual pixels.

**Presenter prompt:** point at the red object and ask which of `pixels[80, 60]` and `pixels[60, 80]` means x=60, y=80.


## Step 2 — Inspect channels and crop an object
Pixel values are integers from 0 to 255. Neural networks commonly receive floating-point values after preprocessing; the precise transform belongs to the model. Cropping follows `[y1:y2, x1:x2]`.


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for channel, name in enumerate(["Red", "Green", "Blue"]):
    axes[channel].imshow(pixels[:, :, channel], cmap="gray", vmin=0, vmax=255)
    axes[channel].set_title(name + " intensity")
x1, y1, x2, y2 = boxes[0].astype(int)
axes[3].imshow(pixels[y1:y2, x1:x2])
axes[3].set_title("Rectangle crop")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()
float_image = pixels.astype(np.float32) / 255.0
print("Float range:", float_image.min(), "to", float_image.max())


## Step 3 — Convert corner boxes to normalized center boxes
A common YOLO annotation row is `class_id center_x center_y width height`, with all four geometry values divided by the image width or height. IDs start at zero. This is a **label format**, not a prediction confidence score.

A normalized x value of 0.5 is halfway across **this image**. Keep image dimensions with your annotations.


In [ ]:
def xyxy_to_normalized_xywh(boxes, width, height):
    boxes = np.asarray(boxes, dtype=np.float32).reshape(-1, 4)
    centers = (boxes[:, :2] + boxes[:, 2:]) / 2
    sizes = boxes[:, 2:] - boxes[:, :2]
    return np.concatenate([centers, sizes], axis=1) / [width, height, width, height]

def normalized_xywh_to_xyxy(rows, width, height):
    rows = np.asarray(rows, dtype=np.float32).reshape(-1, 4)
    scaled = rows * [width, height, width, height]
    centers, sizes = scaled[:, :2], scaled[:, 2:]
    return np.concatenate([centers - sizes / 2, centers + sizes / 2], axis=1)

width, height = image.size
normalized = xyxy_to_normalized_xywh(boxes, width, height)
for class_id, row in enumerate(normalized):
    print(class_id, " ".join(f"{value:.6f}" for value in row))
restored = normalized_xywh_to_xyxy(normalized, width, height)
assert np.allclose(boxes, restored, atol=1e-4)
print("Round-trip conversion preserved the boxes.")


## Step 4 — Resize the picture and its annotations together
Here we deliberately stretch the image to a square. Multiply x coordinates by `new_width / old_width` and y coordinates by `new_height / old_height`. Many real detectors preserve aspect ratio with padding (“letterboxing”); then you must account for both scale and padding.


In [ ]:
new_width, new_height = 256, 256
resized = image.resize((new_width, new_height))
resized_boxes = boxes * [new_width / width, new_height / height,
                        new_width / width, new_height / height]
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
show_boxes(resized, boxes, names, ax=axes[0], title="Mistake: old pixel coordinates")
show_boxes(resized, resized_boxes, names, ax=axes[1], title="Correct: scaled coordinates")
plt.tight_layout()
plt.show()


## Step 5 — Flip horizontally without losing the objects
A horizontal flip maps a half-open box to `[W-x2, y1, W-x1, y2]`. Read from the original box while constructing the new one; in-place edits can accidentally reuse a changed coordinate.


In [ ]:
flipped = image.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
flipped_boxes = boxes.copy()
flipped_boxes[:, 0] = width - boxes[:, 2]
flipped_boxes[:, 2] = width - boxes[:, 0]
show_boxes(flipped, flipped_boxes, names, title="Horizontal flip with correct labels")
plt.show()
assert np.allclose(flipped_boxes[:, 2] - flipped_boxes[:, 0], boxes[:, 2] - boxes[:, 0])


## Try it yourself
1. Add a third object with a valid box; display all three labels.
2. Print the rectangle's area and the fraction of the image it occupies.
3. Derive the box transform for a **vertical** flip. Plot it before opening the solution.

**Checkpoint:** explain why changing an image without changing its annotation silently damages training data.


## Solutions — run after attempting the exercises
The third object below is a triangle. We draw its extreme pixels inside its half-open bounding box.


In [ ]:
extra_image = image.copy()
ImageDraw.Draw(extra_image).polygon([(155, 185), (185, 185), (170, 214)], fill=(35, 160, 90))
extra_boxes = np.vstack([boxes, [155, 185, 186, 215]])
show_boxes(extra_image, extra_boxes, names + ["triangle"])
plt.show()
area = (boxes[0, 2] - boxes[0, 0]) * (boxes[0, 3] - boxes[0, 1])
print("Rectangle box area:", area, "pixels squared; fraction:", area / (width * height))
vertical_boxes = boxes.copy()
vertical_boxes[:, 1] = height - boxes[:, 3]
vertical_boxes[:, 3] = height - boxes[:, 1]
show_boxes(image.transpose(Image.Transpose.FLIP_TOP_BOTTOM), vertical_boxes, names)
plt.show()


## Recap and next step
You can now interpret RGB arrays, crop objects, convert annotations, and transform boxes with an image. Inspect annotations visually every time you change preprocessing.

**Next:** [02 · Classical object detection](02_Classical_Object_Detection.ipynb).

**Reference:** [Pillow image operations](https://pillow.readthedocs.io/en/stable/reference/Image.html).
